<a href="https://colab.research.google.com/github/faisu6339-glitch/LLMs/blob/main/Langchain(Prompt_Engineering).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Basic PromptTemplate
What you learn

You create a reusable prompt containing a variable:

Explain {topic} in simple language.

Then LangChain replaces {topic} with your actual value.

In [9]:
!pip install -q -U langchain langchain-google-genai python-dotenv

In [10]:
!pip install -q -U langchain langchain-google-genai

In [11]:
!pip install -U google-genai langchain-google-genai langchain-core

In [12]:
import os
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in Colab Secrets.")

api_key = api_key.strip()

os.environ["GOOGLE_API_KEY"] = api_key

print("API key loaded successfully.")
print("Key length:", len(api_key))
print("Key prefix:", api_key[:5])

API key loaded successfully.
Key length: 53
Key prefix: AQ.Ab


In [13]:
import os
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")

In [14]:
from google import genai

client = genai.Client(
    api_key=api_key
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello in one sentence."
)

print(response.text)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 6h1m30.649569115s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'globa

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI

#model Evaluation
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.4
)

#1. Without using reusable template:

In [16]:
response = llm.invoke(
    "Explain Machine Learning in simple language."
)

print(response.content)

GoogleRateLimitError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 6h5.603299741s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 

#2.With using reusable template:

# a.Test single topics

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt=PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple language with examples and cnn program."

)
topic="Deep Learning"

final_prompt=prompt.format(topic=topic)

response=llm.invoke(final_prompt)
print(response.content)

# b.Test multiple topics

In [ ]:
topics = [
    "Machine Learning",
    "RNN",
    "Transformer"
]
for topic in topics:
  final_prompt=prompt.format(topic=topic)
  response=llm.invoke(final_prompt)
  print("\nTOPIC:",topic)
  print(response.content)

# c. Multiple Variables
Now we have more than one variable.

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["topic", "level", "language"],
    template="Explain {topic} for a {level} student in {language}."
)
topic = "Transformer"
level = "beginner"
language = "Hindi"
final_prompt = prompt.format(
    topic=topic,
    level=level,
    language=language
)

print(final_prompt)

In [ ]:
response = llm.invoke(final_prompt)

print(response.content)

questions = [
    {
        "topic": "Machine Learning",
        "level": "beginner",
        "language": "English"
    },
    {
        "topic": "RNN",
        "level": "intermediate",
        "language": "Hindi"
    },
    {
        "topic": "Transformer",
        "level": "advanced",
        "language": "English"
    }
]

for q in questions:

    final_prompt = prompt.format(
        topic=q["topic"],
        level=q["level"],
        language=q["language"]
    )

    response = llm.invoke(final_prompt)

    print("\n--------------------")
    print(final_prompt)
    print("--------------------")
    print(response.content)

#d.Multiple variables different topics

In [ ]:
cars="BMW"
bikes="Ducati"
trucks="Ashok Leyland"

In [ ]:
prompt_vehicles = PromptTemplate(
    input_variables=["cars", "bikes", "trucks"],
    template="Explain me price of {cars} milege of {bikes} and capacity of {trucks}"
)

final_prompt = prompt_vehicles.format(cars=cars, bikes=bikes, trucks=trucks)
print(final_prompt)

response = llm.invoke(final_prompt)
print(response.content)

# 2.ChatPromptTemplate
Now we move from a simple text prompt to a chat structure.

We usually have:
System

   ↓

User

   ↓
   
Gemini

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_messages([
    ("system","You are an expert AI teacher and very good explainer"),
    ("user","Explain {topic} in simple Language.")
])

In [ ]:
#Now provide the topic:
message = prompt.format_prompt(topic="Machine Learning").to_messages()
print(message)

In [ ]:
#Send to Gemini:
response = llm.invoke(message)

print(response.content)

Understanding System vs User
Here:
("system", "You are an expert AI teacher.")


means:
Gemini should behave like an AI teacher.

And:
("user", "Explain {topic} in simple language.")


is the actual user request.

# a.Few-Shot Prompt
Now you're teaching Gemini using examples.

For example:

1.Positive → Happy

2.Negative → Sad

Amazing movie → ?

We want:
Happy

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Classify the sentiment.

Example 1:
Input: I love this movie.
Output: Positive

Example 2:
Input: This movie is terrible.
Output: Negative

Example 3:
Input: The movie is okay.
Output: Neutral

Now classify:

Input: {text}

Output:
"""
)

In [ ]:
#Text
text="Amazing movie"
final_prompt=prompt.format(text=text)
print(final_prompt)

In [ ]:
#Send to Gemini
response=llm.invoke(final_prompt)
print(response.content)

# b.Try multiple examples

In [ ]:
texts = [
    "I absolutely love this product.",
    "This is the worst service.",
    "The product is okay.",
    "The movie was fantastic.",
    "I did not enjoy the movie."
]

for text in texts:

    final_prompt = prompt.format(text=text)

    response = llm.invoke(final_prompt)

    print("Input:", text)
    print("Output:", response.content)
    print("----------------------")

# 3. Prompt for Classification
Now let's build a proper sentiment classifier.

Possible outputs:

1.**Positive**
2.**Negative**
3.**Neutral**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a sentiment classifier.

        Classify the input into exactly one of:

        Positive
        Negative
        Neutral

        Return only the category.
        """
    ),

    (
        "user",
        "Text: {text}"
    )
])

In [ ]:
# Test:
text="I absolutely love this product."
final_prompt=prompt.format_prompt(text=text).to_messages()
print(final_prompt)

response=llm.invoke(final_prompt)
print(response.content)

#4. Prompt for Translation

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Translate the following English sentence into Hindi.

English:
{text}

Hindi:
"""
)

In [ ]:
text = "Machine learning is changing the world."

final_prompt = prompt.format(text=text)

response = llm.invoke(final_prompt)

print(response.content)

# 5.Prompt for Summarization

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Summarize the following text.

Text:
{text}

Summary:
"""
)

In [ ]:
text = """
Machine learning is a branch of artificial intelligence that allows
computers to learn patterns from data without being explicitly programmed.
Machine learning is widely used in recommendation systems, fraud detection,
image classification, speech recognition, healthcare, finance and many other
applications. There are three major types of machine learning:
supervised learning, unsupervised learning and reinforcement learning.
"""

In [ ]:
final_prompt=prompt.format(text=text)
print(final_prompt)

response=llm.invoke(final_prompt)
print(response.content)

# a.Short Summary

In [ ]:
prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Give a very short summary of the following text in 2-3 sentences.

Text:
{text}

Short Summary:
"""
)

In [ ]:
response = llm.invoke(
    prompt.format(text=text)
)

print(response.content)

# b.Detailed Summary

In [ ]:
prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Give a detailed summary of the following text.

Include:
- Main idea
- Important concepts
- Important details
- Applications

Text:
{text}

Detailed Summary:
"""
)

In [ ]:
response = llm.invoke(
    prompt.format(text=text)
)

print(response.content)

# c.Bullet Summary

In [ ]:
prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Summarize the following text using bullet points.

Text:
{text}

Bullet Summary:
"""
)

In [ ]:
response = llm.invoke(
    prompt.format(text=text)
)

print(response.content)